**Klasik kütüphanelere ek olarak, modelin kararlarını görselleştirmek ve istatistiksel dönüşümler yapmak için gerekli modülleri yüklüyoruz.**

In [3]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# 🔥 DÜZELTME: IterativeImputer için deneysel özellikleri açıyoruz
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer, KNNImputer

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import HuberRegressor
from sklearn.cluster import KMeans
import lightgbm as lgb
from catboost import CatBoostRegressor

# Görsel ve sistem ayarları
warnings.filterwarnings('ignore')
sns.set_palette("muted")
sns.set_style("whitegrid")

**Neden Yapıyoruz? Ağaç tabanlı modeller (Gradient Boosting) hedef değişkendeki çarpıklığa (skewness) karşı hassastır. "Bilişsel Performans Skoru"nun varyansını sabitlemek ve normal dağılıma yaklaştırmak için Yeo-Johnson güç dönüşümü (power transformation) kullanıyoruz. Bu, modelin kuyruk (uç) değerlerdeki hata payını (RMSE) dramatik şekilde düşürür.**

In [6]:
print("1. Veriler Yükleniyor ve Hedef Varyans Stabilizasyonu Uygulanıyor...")
path = '/kaggle/input/competitions/yzta-2026-datathon/'
train = pd.read_csv(path + 'train.csv')
test = pd.read_csv(path + 'test_x.csv')

hedef = 'bilissel_performans_skoru'
y_saf = train[hedef].values

# Varyans Sabitleme ve Normalleştirme
ptransform = PowerTransformer(method='yeo-johnson')
y_transformed = ptransform.fit_transform(y_saf.reshape(-1, 1)).ravel()

train_id, test_id = train['id'], test['id']
train.drop(['id', hedef], axis=1, inplace=True)
test.drop(['id'], axis=1, inplace=True)

1. Veriler Yükleniyor ve Hedef Varyans Stabilizasyonu Uygulanıyor...



**K-Means ile "Latent Space" (Gizli Profil) KeşfiNeden Yapıyoruz? İnsan sağlığı çok değişkenlidir. Yaşı 50 olan ve stresi 8 olan biriyle, yaşı 20 olup stresi 8 olan birinin bilişsel performansı aynı tepkiyi vermez. K-Means kullanarak, veri setindeki bu doğrusal olmayan (non-linear) çok boyutlu ilişkileri, tek boyutlu bir "Arketip/Persona" sınıfına (Latent Feature) indirgiyoruz. Böylece algoritmalarımıza yön bulmaları için güçlü bir 'sezgi' (heuristic) veriyoruz.******


In [8]:
from sklearn.impute import SimpleImputer # 🔥 Güvenlik duvarı için eklendi

print("2. Unsupervised Learning: K-Means ile Psikolojik Personalar Keşfediliyor...")

# İnsan biyolojisini tanımlayan çekirdek metrikler
km_features = ['yas', 'vucut_kitle_indeksi', 'stres_skoru', 'derin_uyku_yuzdesi', 'gunluk_adim_sayisi']

# 🔥 KESİN ÇÖZÜM: K-Means'in çökmesini engellemek için sızan NaN değerleri temizliyoruz
imputer_km = SimpleImputer(strategy='median')
train_km_clean = imputer_km.fit_transform(train[km_features])
test_km_clean = imputer_km.transform(test[km_features])

# Mesafeye dayalı algoritmalar (K-Means) için standardizasyon zorunludur
scaler = StandardScaler()
train_km_scaled = scaler.fit_transform(train_km_clean)
test_km_scaled = scaler.transform(test_km_clean)

# Veriyi 5 eşsiz arketipe ayırıyoruz
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
train['psikolojik_profil'] = kmeans.fit_predict(train_km_scaled).astype(str)
test['psikolojik_profil'] = kmeans.predict(test_km_scaled).astype(str)

print(" -> 5 Eşsiz Psikolojik Profil başarıyla Latent Space'ten çıkarıldı ve NaN hatası aşıldı!")

2. Unsupervised Learning: K-Means ile Psikolojik Personalar Keşfediliyor...
 -> 5 Eşsiz Psikolojik Profil başarıyla Latent Space'ten çıkarıldı ve NaN hatası aşıldı!



**Özellik Mühendisliği (Domain Expertise)Neden Yapıyoruz? Sadece ham veriye güvenmek yerine, klinik/psikolojik literatürden yola çıkarak "Tükenmişlik (Burnout)", "Uyku Verimliliği" ve "Hareket/Stres Oranı" gibi türetilmiş, yoğunluğu yüksek sinyaller oluşturuyoruz.**


In [11]:
print("3. Özellik Mühendisliği & Etkileşim (Interaction) Sütunları...")

def create_advanced_features(df):
    df_temp = df.copy()
    # Klinik tükenmişlik indeksi (Stres yükünün uyku dinlenmesine oranı)
    df_temp['tukenmislik_indeksi'] = (df_temp['stres_skoru'] * df_temp['gunluk_calisma_saati']) / (df_temp['derin_uyku_yuzdesi'] + 1)
    
    # Uyku Konsolidasyon Oranı
    df_temp['uyku_verimliligi'] = (df_temp['derin_uyku_yuzdesi'] + df_temp['rem_yuzdesi']) / (df_temp['uykuya_dalma_suresi_dk'] + 1)
    
    # Otonom Sinir Sistemi Yükü
    df_temp['otonom_stres_yuku'] = df_temp['stres_skoru'] * df_temp['vucut_kitle_indeksi']
    return df_temp

train = create_advanced_features(train)
test = create_advanced_features(test)

# Kategorik değişkenleri One-Hot Encoding ile makine diline çeviriyoruz
categorical_cols = ['cinsiyet', 'kronotip', 'mevsim', 'psikolojik_profil']

# 🔥 KESİN ÇÖZÜM: Sütunlar hala var mı diye kontrol et (Hücreyi iki kere çalıştırma hatasını önler)
mevcut_kategorik = [col for col in categorical_cols if col in train.columns]

if mevcut_kategorik:
    train = pd.get_dummies(train, columns=mevcut_kategorik, drop_first=True)
    test = pd.get_dummies(test, columns=mevcut_kategorik, drop_first=True)
    print(f" -> {mevcut_kategorik} sütunları başarıyla One-Hot formata çevrildi.")
else:
    print(" -> Kategorik değişkenler zaten çevrilmiş (Hücre tekrar çalıştırılmış olabilir, sorun yok).")

3. Özellik Mühendisliği & Etkileşim (Interaction) Sütunları...
 -> Kategorik değişkenler zaten çevrilmiş (Hücre tekrar çalıştırılmış olabilir, sorun yok).



**Açıklanabilir Yapay Zeka (XAI) ile Cerrahi Özellik BudamasıNeden Yapıyoruz? Çok fazla sütun (özellik), modelin gereksiz gürültüyü (noise) ezberlemesine neden olur. Bir LightGBM Değerlendirici Ajanı kurarak her bir sütunun "Information Gain" (Bilgi Kazancı) skorunu ölçüyoruz. Modelin karar mekanizmasına katkısı %0'a yakın olan "ölü yükleri" tespit edip acımasızca siliyoruz. Bu, test setindeki genelleme (generalization) yeteneğini maksimize eder.**


In [14]:
print("4. Cerrahi Budama: Bilgi Kazancı (Information Gain) Analizi...")

# Sayısal eksikleri geçici olarak KNN ile dolduralım (Sadece analiz için)
imputer_temp = KNNImputer(n_neighbors=3)
train_temp = pd.DataFrame(imputer_temp.fit_transform(train.select_dtypes(include=[np.number])), columns=train.select_dtypes(include=[np.number]).columns)

# Değerlendirici Ajan
evaluator = lgb.LGBMRegressor(n_estimators=300, importance_type='gain', random_state=42, verbose=-1)
evaluator.fit(train_temp, y_transformed)

# Özellik Önem Derecelerini Çıkarma
feature_imp = pd.DataFrame({'Ozellik': train_temp.columns, 'Gain': evaluator.feature_importances_})
feature_imp = feature_imp.sort_values(by='Gain', ascending=False).reset_index(drop=True)

# En alttaki %20'lik dilimi (Ölü Yük) kesiyoruz
silinecek_yuzde = int(len(train_temp.columns) * 0.20)
olu_yukler = feature_imp.tail(silinecek_yuzde)['Ozellik'].tolist()

train.drop(columns=olu_yukler, inplace=True, errors='ignore')
test.drop(columns=olu_yukler, inplace=True, errors='ignore')
print(f" -> Kesilen Ölü Yük Sütun Sayısı: {len(olu_yukler)}")

4. Cerrahi Budama: Bilgi Kazancı (Information Gain) Analizi...
 -> Kesilen Ölü Yük Sütun Sayısı: 3


Meta-Modelleme: Huber Robust Stacking & 5-Seed Averaging
Neden Yapıyoruz? 1. 5-Seed Averaging: Başlangıç noktasındaki rastgelelikten (randomness) kurtulmak için modelleri 5 farklı 'seed' ile eğitip ortalamasını alıyoruz. Varyansı düşürür.
2. Huber Loss (Robust Stacking): OOF tahminlerini birleştirirken standart Ridge (L2 cezası) yerine Huber Regressor kullanıyoruz. Huber, küçük hatalara RMSE gibi, büyük (aykırı) hatalara MAE gibi davranır. Bu zırh, Private Leaderboard'da (Gizli Liderlik Tablosu) yaşanacak büyük sıralama depremlerinden (shake-up) bizi korur.

In [16]:
print("5. 5-Seed Averaging ve Huber Robust Stacking Başlıyor...")

# =====================================================================================
# 🔥 DEMİR KUBBE FİLTRESİ: Modeli çökertecek tüm metin/bool sütunlarını dışarıda bırakıyoruz.
# Sadece algoritmaların okuyabildiği sayısal (np.number) sütunlar eğitime alınır.
# =====================================================================================
train_num = train.select_dtypes(include=[np.number])
test_num = test.select_dtypes(include=[np.number])

# Test ve Train setinin sütun sayılarını/sıralarını birebir eşitliyoruz (Olası boyut çökmesini engeller)
train_num, test_num = train_num.align(test_num, join='inner', axis=1)

SEEDS = [42, 2026, 999, 1071, 1453] 
final_predictions = np.zeros(len(test_num))

# (Not: Optuna ile bulunmuş optimal hiperparametreler kullanılmıştır)
for seed in SEEDS:
    print(f"--- SEED {seed} ---")
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    
    oof_features = np.zeros((len(train_num), 2))
    test_features = np.zeros((len(test_num), 2))
    
    for fold, (trn_idx, val_idx) in enumerate(kf.split(train_num)):
        # 🔥 Artık 'train' yerine sadece filtrelenmiş 'train_num' kullanıyoruz
        X_tr, X_val = train_num.iloc[trn_idx], train_num.iloc[val_idx]
        y_tr, y_val = y_transformed[trn_idx], y_transformed[val_idx]
        
        # LGBM ve CatBoost Eğitimleri 
        model_lgb = lgb.LGBMRegressor(objective='huber', alpha=1.25, n_estimators=1500, learning_rate=0.015, random_state=seed, verbose=-1)
        model_cat = CatBoostRegressor(loss_function='Huber:delta=1.25', iterations=1500, learning_rate=0.02, random_seed=seed, verbose=0)
        
        model_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
        model_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100)
        
        oof_features[val_idx, 0] = model_lgb.predict(X_val)
        oof_features[val_idx, 1] = model_cat.predict(X_val)
        
        # Tahminlerde de filtrelenmiş 'test_num' kullanıyoruz
        test_features[:, 0] += model_lgb.predict(test_num) / 5
        test_features[:, 1] += model_cat.predict(test_num) / 5
        
    # Aykırı değerlere dirençli Meta-Model (HuberRegressor)
    meta_model = HuberRegressor(epsilon=1.35, alpha=10.0)
    meta_model.fit(oof_features, y_transformed)
    
    # Tohum (Seed) bazlı tahminleri ters dönüşümle (Inverse Transform) alıp topluyoruz
    seed_preds_transformed = meta_model.predict(test_features)
    final_predictions += ptransform.inverse_transform(seed_preds_transformed.reshape(-1, 1)).ravel() / len(SEEDS)

print("\nModel Mimarisi Tamamlandı. Güvenli Tahminler Üretildi.")

5. 5-Seed Averaging ve Huber Robust Stacking Başlıyor...
--- SEED 42 ---
--- SEED 2026 ---
--- SEED 999 ---
--- SEED 1071 ---
--- SEED 1453 ---

Model Mimarisi Tamamlandı. Güvenli Tahminler Üretildi.


Submission: Liderlik Tablosuna Hazırlık
Test setinde modelin matematiksel olarak imkansız değerler üretmesini engellemek için tahminleri np.clip ile 0-10 aralığına zorluyoruz.

In [17]:
print("6. Gönderim (Submission) Dosyası Çıkarılıyor...")

submission_df = pd.DataFrame({
    'id': test_id,
    'bilissel_performans_skoru': np.clip(final_predictions, 0, 10)
})

submission_df.to_csv('submission.csv', index=False)
print("🎯 Grandmaster Pipeline Tamamlandı: 'submission.csv' 24000x2 formatında hazır!")

6. Gönderim (Submission) Dosyası Çıkarılıyor...
🎯 Grandmaster Pipeline Tamamlandı: 'submission.csv' 24000x2 formatında hazır!
